# 01 - Lancer le traitement compliance NLP

Ce notebook execute le traitement local sur les PDF du dossier `data/raw` et genere les resultats dans `outputs/analysis/results.json`.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
elif not (repo_root / "src").exists() and (repo_root.parent / "src").exists():
    repo_root = repo_root.parent

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

repo_root

In [ ]:
from compliance_nlp import (
    analyze_directory,
    load_article9_terms,
    load_forbidden_terms,
    load_whitelist_terms,
    results_to_dataframe,
    summarize_results,
)

import pandas as pd

## Parametres d'execution

In [ ]:
input_dir = repo_root / "data" / "raw"
results_path = repo_root / "outputs" / "analysis" / "results.json"
findings_csv_path = repo_root / "outputs" / "analysis" / "findings.csv"

forbidden_words_path = repo_root / "configs" / "forbidden_words.csv"
article9_terms_path = repo_root / "configs" / "article9_terms.csv"
whitelist_path = repo_root / "configs" / "article9_whitelist.csv"

params = {
    "input_dir": str(input_dir),
    "results_path": str(results_path),
    "findings_csv_path": str(findings_csv_path),
    "forbidden_words_path": str(forbidden_words_path),
    "article9_terms_path": str(article9_terms_path),
    "whitelist_path": str(whitelist_path),
}
params

## Verification des fichiers d'entree

In [ ]:
pdf_files = sorted(input_dir.glob("*.pdf"))
pd.DataFrame({"pdf": [pdf.name for pdf in pdf_files], "size_bytes": [pdf.stat().st_size for pdf in pdf_files]})

## Verification des referentiels de controle

In [ ]:
forbidden_terms = load_forbidden_terms(forbidden_words_path)
article9_terms = load_article9_terms(article9_terms_path)
whitelist_terms = load_whitelist_terms(whitelist_path)

pd.DataFrame([
    {"referentiel": "forbidden_words", "count": len(forbidden_terms)},
    {"referentiel": "article9_terms", "count": len(article9_terms)},
    {"referentiel": "article9_whitelist", "count": len(whitelist_terms)},
])

In [ ]:
pd.DataFrame([
    {
        "rule_id": term.rule_id,
        "category": term.category,
        "label": term.label,
        "terms": " | ".join(term.terms),
        "synonyms": " | ".join(term.synonyms),
        "base_score": term.base_score,
        "fuzzy_threshold": term.fuzzy_threshold,
    }
    for term in article9_terms
])

## Execution du traitement

In [ ]:
results = analyze_directory(
    input_dir,
    output_path=results_path,
    forbidden_words_path=forbidden_words_path,
    article9_terms_path=article9_terms_path,
    whitelist_path=whitelist_path,
)

summary = summarize_results(results)
summary

## Resultats generes

In [ ]:
df = results_to_dataframe(results)
findings_csv_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(findings_csv_path, index=False, encoding="utf-8-sig")

{
    "json_results": str(results_path),
    "csv_findings": str(findings_csv_path),
    "rows": len(df),
}

In [ ]:
df[[
    "document_name",
    "code",
    "section",
    "category",
    "matched_term",
    "detection_type",
    "score",
    "severity",
    "alert_level",
]].sort_values(["document_name", "score"], ascending=[True, False], na_position="last")